In [3]:
# ============================================================================
# IMPORTS
# ============================================================================
import re          # Regular expressions for pattern matching
import json        # JSON serialization
import argparse    # Command-line argument parsing
import sys         # System utilities
from pathlib import Path              # Modern path handling
from typing import List, Dict, Optional  # Type hints

In [8]:
def extract_chunks_from_markdown(markdown_text: str) -> List[Dict]:
    """
    Extract all chunks from markdown with boundary markers

    This is the foundation - extracts atomic chunks from boundary markers.

    Args:
        markdown_text: Markdown content with boundary markers

    Returns:
        List of atomic chunk dictionaries
    """
    """
    <!-- BOUNDARY_START type="header" id="p1_header_1" page="1" level="1" breadcrumbs="Thematics" -->
        ## Thematics
    <!-- BOUNDARY_END type="header" id="p1_header_1" -->
    <!-- BOUNDARY_START type="header" id="p1_header_2" page="1" level="1" breadcrumbs="Uncovering Alpha in AI's Rate of Change" -->
        ## Uncovering Alpha in AI's Rate of Change
    <!-- BOUNDARY_END type="header" id="p1_header_2" -->
    [
        ('"type="header" id="p1_header_1" page="1" level="1" breadcrumbs="Thematics"','## Thematics','type="header" id="p1_header_1"'),
        ('type="header" id="p1_header_2" page="1" level="1" breadcrumbs="Uncovering Alpha in AI's Rate of Change"','## Uncovering Alpha in AI's Rate of Change','type="header" id="p1_header_2"')
    ]
    """
    # Pattern to match boundary markers and content between them
    pattern = r'<!-- BOUNDARY_START (.*?) -->\n(.*?)\n<!-- BOUNDARY_END (.*?) -->'

    # Find all matches (DOTALL flag to match across newlines)
    matches = re.findall(pattern, markdown_text, re.DOTALL)
    chunks = []

    for start_attrs, content, end_attrs in matches:
        # Parse attributes from START marker
        attrs = dict(re.findall(r'(\w+)="([^"]*)"', start_attrs))

        # Extract core fields
        chunk = {
            'id': attrs.get('id', 'unknown'),
            'type': attrs.get('type', 'unknown'),
            'page': attrs.get('page', '0'),
            'content': content.strip()
        }

        # Add remaining attributes as metadata
        metadata = {k: v for k, v in attrs.items()
                   if k not in ['id', 'type', 'page']}

        if metadata:
            chunk['metadata'] = metadata

        chunks.append(chunk)

    return chunks

In [9]:
def chunk_directory(dir_path: Path) -> Dict[str, List[Dict]]:
    """
    Extract chunks from all markdown files in directory

    Automatically handles:
    - Direct pages directory: extracted_docs/doc_name/pages/
    - Parent directory: extracted_docs/doc_name/
    - Batch directory: extracted_docs/
    """
    results = {}

    # Check if directory contains .md files directly
    md_files = list(dir_path.glob("*.md"))

    if md_files:
        # This is a pages directory
        for md_file in sorted(md_files):
            chunks = chunk_file(md_file)
            results[md_file.name] = chunks
    else:
        # Look for pages/ subdirectories
        pages_dir = dir_path / "pages"
        if pages_dir.exists() and pages_dir.is_dir():
            # Single document with pages/ subdirectory
            for md_file in sorted(pages_dir.glob("*.md")):
                chunks = chunk_file(md_file)
                results[md_file.name] = chunks
        else:
            # Multiple documents with */pages/ pattern
            for pages_dir in dir_path.glob("*/pages"):
                if pages_dir.is_dir():
                    for md_file in sorted(pages_dir.glob("*.md")):
                        chunks = chunk_file(md_file)
                        key = f"{pages_dir.parent.name}/{md_file.name}"
                        results[key] = chunks

    return results

In [10]:
def chunk_file(file_path: Path) -> List[Dict]:
    """Extract chunks from a single markdown file"""
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    return extract_chunks_from_markdown(content)

In [16]:
actomic_file_chunks = chunk_directory(Path('/Users/akellaprudhvi/mystuff/Course/GenAI-Course-Modules/Module_4/extraction/extracted_docs_bounded/AI-Enablers-Adopters-research-report'))

In [20]:
actomic_chunks = []
for index,file_chunks in  enumerate(actomic_file_chunks.values()):
    print(f"pagenumber:{index}")
    actomic_chunks.extend(file_chunks)
    print("---------")

pagenumber:0
---------
pagenumber:1
---------
pagenumber:2
---------
pagenumber:3
---------
pagenumber:4
---------
pagenumber:5
---------
pagenumber:6
---------


In [23]:
def create_semantic_chunks(
        chunks: List[Dict],
        target_size: int = 1500,
        min_size: int = 800,
        max_size: int = 3000,
        max_table_size: int = 2000
) -> List[Dict]:
    """
    Create semantic chunks with intelligent type-aware grouping

    KEY IMPROVEMENTS OVER BASIC CHUNKING:
    ======================================
    1. Headers ALWAYS bind to following content (no orphaned headers)
    2. Large tables (>max_table_size) become standalone chunks
    3. Look-ahead logic prevents flushing before headers
    4. Smart section boundary detection

    Args:
        chunks: List of atomic chunks from boundary markers
        target_size: Target character count (soft target, default 1500)
        min_size: Minimum size before flushing (default 800)
        max_size: Hard maximum size (default 3000)
        max_table_size: Tables larger than this become standalone (default 2000)

    Returns:
        List of semantic chunks with better coherence
    """

    semantic_chunks = []
    buffer = []
    buffer_size = 0
    current_breadcrumb = None

    # -----------------------------------------------------------------------
    # Helper: Check if next chunk is a header
    # -----------------------------------------------------------------------
    def next_is_header(current_idx: int) -> bool:
        """Check if the next chunk in sequence is a header"""
        if current_idx + 1 < len(chunks):
            return chunks[current_idx + 1]['type'] == 'header'
        return False

    # -----------------------------------------------------------------------
    # Helper: Flush buffer to create semantic chunk
    # -----------------------------------------------------------------------
    def flush_buffer():
        """Create semantic chunk from current buffer and reset"""
        nonlocal buffer, buffer_size, semantic_chunks

        if buffer:
            semantic_chunks.append({
                'combined_content': '\n\n'.join([c['content'] for c in buffer]),
                'chunk_ids': [c['id'] for c in buffer],
                'breadcrumbs': current_breadcrumb,
                'char_count': buffer_size,
                'num_chunks': len(buffer),
                'chunk_types': [c['type'] for c in buffer]
            })
            buffer = []
            buffer_size = 0

    # -----------------------------------------------------------------------
    # Main chunking loop
    # -----------------------------------------------------------------------
    for idx, chunk in enumerate(chunks):
        chunk_type = chunk['type']
        breadcrumb = chunk.get('metadata', {}).get('breadcrumbs', '')
        chunk_size = len(chunk['content'])

        # -------------------------------------------------------------------
        # SPECIAL CASE: Large tables become standalone chunks
        # -------------------------------------------------------------------
        if chunk_type == 'table' and chunk_size > max_table_size:
            # Flush current buffer first
            flush_buffer()

            # Make table its own chunk
            semantic_chunks.append({
                'combined_content': chunk['content'],
                'chunk_ids': [chunk['id']],
                'breadcrumbs': breadcrumb,
                'char_count': chunk_size,
                'num_chunks': 1,
                'chunk_types': ['table']
            })
            current_breadcrumb = breadcrumb
            continue

        # -------------------------------------------------------------------
        # Check flush conditions (with type awareness!)
        # -------------------------------------------------------------------
        section_changed = (current_breadcrumb and breadcrumb != current_breadcrumb)
        would_exceed_max = (buffer_size + chunk_size > max_size)
        is_next_header = next_is_header(idx)

        # -------------------------------------------------------------------
        # RULE 1: Section changed + big enough + NOT before header
        # -------------------------------------------------------------------
        if section_changed and buffer_size >= min_size and not is_next_header:
            flush_buffer()

        # -------------------------------------------------------------------
        # RULE 2: Would exceed max_size + big enough
        # -------------------------------------------------------------------
        elif would_exceed_max and buffer_size >= min_size:
            flush_buffer()
            # if chunk_type != 'header':
            #     flush_buffer()
            # else:
            #     # Header will start new buffer - flush old one
            #     flush_buffer()

        # -------------------------------------------------------------------
        # Add current chunk to buffer
        # -------------------------------------------------------------------
        buffer.append(chunk)
        buffer_size += chunk_size
        current_breadcrumb = breadcrumb

        # -------------------------------------------------------------------
        # RULE 3: Buffer reached target size + NOT before header
        # -------------------------------------------------------------------
        if buffer_size >= target_size and not is_next_header:
            flush_buffer()

    # -----------------------------------------------------------------------
    # Flush remaining buffer
    # -----------------------------------------------------------------------
    flush_buffer()

    return semantic_chunks

In [28]:
semantic_chunks = create_semantic_chunks(actomic_chunks)

In [26]:
def format_chunks_for_output(semantic_chunks: List[Dict], keep_ids: bool = False) -> List[Dict]:
    """
    Convert semantic chunks to clean output format with 'content' field

    This is now the ONLY output format - clean and simple!

    Output format:
    {
      "content": "...",
      "metadata": {
        "breadcrumbs": "...",
        "char_count": 1500,
        "num_atomic_chunks": 5
      }
    }

    Args:
        semantic_chunks: Raw semantic chunks from create_semantic_chunks()
        keep_ids: Include chunk_ids in metadata (for debugging)

    Returns:
        List of clean formatted chunks ready for enrichment
    """
    clean = []

    for chunk in semantic_chunks:
        clean_chunk = {
            'content': chunk['combined_content'],  # Always use 'content' field
            'metadata': {
                'breadcrumbs': chunk.get('breadcrumbs', ''),
                'char_count': chunk['char_count'],
                'num_atomic_chunks': chunk['num_chunks']
            }
        }

        # Optionally include chunk_ids for debugging
        if keep_ids:
            clean_chunk['metadata']['chunk_ids'] = chunk['chunk_ids']
            clean_chunk['metadata']['chunk_types'] = chunk['chunk_types']

        clean.append(clean_chunk)

    return clean

In [29]:
cleand_semantic_chunks = format_chunks_for_output(semantic_chunks)

In [32]:
output_path = "semantic_chunks.json"
with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(cleand_semantic_chunks, f, indent=2, ensure_ascii=False)